In [1]:
# ############################################################
# 연관규칙 예시 — 장바구니 분석 (함께 사는 규칙 찾기)
# ############################################################
# ------------------------------------------------------------
# [목적] 도구 + 데이터 준비 (영수증 5장)
#        (mlxtend 미설치 시: pip install mlxtend)
# ------------------------------------------------------------
import pandas as pd                               # 표 다루기
from mlxtend.frequent_patterns import apriori, association_rules  # 빈발조합 / 규칙 뽑기
receipts = [                                      # 영수증 5장 (한 줄 = 한 명의 장바구니)
    ['라면', '계란', '우유'],
    ['라면', '계란'],
    ['라면', '계란', '콜라'],
    ['우유', '빵'],
    ['라면', '콜라'],
]

In [2]:
# ------------------------------------------------------------
# [목적] 표로 바꾸기 — 샀으면 True (품목을 열로)
# ------------------------------------------------------------
items = sorted({i for r in receipts for i in r})  # 등장한 모든 품목 모으기
df = pd.DataFrame([{i: (i in r) for i in items} for r in receipts])  # 영수증마다 품목별 True/False
df                                                # 0/1(True/False) 표 확인

,계란,라면,빵,우유,콜라
0,True,True,False,True,False
1,True,True,False,False,False
2,True,True,False,False,True
3,False,False,True,True,False
4,False,True,False,False,True


In [3]:
# ------------------------------------------------------------
# [목적] 자주 같이 나오는 조합 찾기 (지지도 0.4 이상)
# ------------------------------------------------------------
freq = apriori(df, min_support=0.4, use_colnames=True)  # 전체의 40% 이상 등장한 조합
freq                                              # 빈발 품목/조합 목록

,support,itemsets
0,0.6,frozenset({계란})
1,0.8,frozenset({라면})
2,0.4,frozenset({우유})
3,0.4,frozenset({콜라})
4,0.6,"frozenset({라면, 계란})"
5,0.4,"frozenset({라면, 콜라})"


In [4]:
# ------------------------------------------------------------
# [목적] 규칙 뽑기 — 향상도(lift) 1 이상만 (우연보다 자주 함께)
# ------------------------------------------------------------
rules = association_rules(freq, metric='lift', min_threshold=1.0)   # A -> B 규칙
print(rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].to_string())

       antecedents      consequents  support  confidence  lift
0  frozenset({라면})  frozenset({계란})      0.6        0.75  1.25
1  frozenset({계란})  frozenset({라면})      0.6        1.00  1.25
2  frozenset({라면})  frozenset({콜라})      0.4        0.50  1.25
3  frozenset({콜라})  frozenset({라면})      0.4        1.00  1.25


In [5]:
# ============================================================
# [결과 해석]
#  · 용어: support(지지도)=전체 중 함께 등장 비율 / confidence(신뢰도)=A산 사람 중 B도 산 비율
#          lift(향상도)=우연 대비 몇 배 더 함께 사는가 (1=무관, >1=연관, <1=오히려 덜)
#  · 규칙 예: '계란 -> 라면' 신뢰도 1.00, 향상도 1.25
#      -> 계란 산 사람은 100% 라면도 샀고, 우연보다 1.25배 자주 함께 삼
#  · 이런 규칙으로 '라면 옆에 계란 진열', '계란 사면 라면 추천' 같은 전략을 세움
# ============================================================